# Lab4-Assignment about Named Entity Recognition and Classification

This notebook describes the assignment of Lab 4 of the text mining course. We assume you have succesfully completed Lab1, Lab2 and Lab3 as welll. Especially Lab2 is important for completing this assignment.

**Learning goals**
* going from linguistic input format to representing it in a feature space
* working with pretrained word embeddings
* train a supervised classifier (SVM)
* evaluate a supervised classifier (SVM)
* learn how to interpret the system output and the evaluation results
* be able to propose future improvements based on the observed results


## Credits
This notebook was originally created by [Marten Postma](https://martenpostma.github.io) and [Filip Ilievski](http://ilievski.nl) and adapted by Piek vossen

## [Points: 18] Exercise 1 (NERC): Training and evaluating an SVM using CoNLL-2003

**[4 point] a) Load the CoNLL-2003 training data using the *ConllCorpusReader* and create for both *train.txt* and *test.txt*:**

    [2 points]  -a list of dictionaries representing the features for each training instances, e..g,
    ```
    [
    {'words': 'EU', 'pos': 'NNP'}, 
    {'words': 'rejects', 'pos': 'VBZ'},
    ...
    ]
    ```

    [2 points] -the NERC labels associated with each training instance, e.g.,
    dictionaries, e.g.,
    ```
    [
    'B-ORG', 
    'O',
    ....
    ]
    ```

In [1]:
from nltk.corpus.reader import ConllCorpusReader

train = ConllCorpusReader(r'CONLL2003/CONLL2003', 'train.txt', ['words', 'pos', 'ignore', 'chunk'])

training_features = []
training_gold_labels = []

for token, pos, ne_label in train.iob_words():
    training_features.append({'words': token, 'pos': pos})
    training_gold_labels.append(ne_label)

print(len(training_features), 'training instances')
print('first 5 features:', training_features[:5])
print('first 5 labels:  ', training_gold_labels[:5])


c:\Users\nicol\anaconda3\envs\text_mining\lib\site-packages\nltk\data.py:388: RuntimeWarning: Security Warning [pathsec.open]: Path C:\Users\nicol\Documents\Group-4-Text-Mining\lab_sessions\lab4\CONLL2003\CONLL2003\train.txt allowed via CWD.
  stream = _secure_open(self._path, "rb")


203621 training instances
first 5 features: [{'words': 'EU', 'pos': 'NNP'}, {'words': 'rejects', 'pos': 'VBZ'}, {'words': 'German', 'pos': 'JJ'}, {'words': 'call', 'pos': 'NN'}, {'words': 'to', 'pos': 'TO'}]
first 5 labels:   ['B-ORG', 'O', 'B-MISC', 'O', 'O']


In [2]:
test = ConllCorpusReader(r'CONLL2003/CONLL2003', 'test.txt', ['words', 'pos', 'ignore', 'chunk'])

test_features = []
test_gold_labels = []

for token, pos, ne_label in test.iob_words():
    test_features.append({'words': token, 'pos': pos})
    test_gold_labels.append(ne_label)

print(len(test_features), 'test instances')
print('first 5 features:', test_features[:5])
print('first 5 labels:  ', test_gold_labels[:5])


c:\Users\nicol\anaconda3\envs\text_mining\lib\site-packages\nltk\data.py:388: RuntimeWarning: Security Warning [pathsec.open]: Path C:\Users\nicol\Documents\Group-4-Text-Mining\lab_sessions\lab4\CONLL2003\CONLL2003\test.txt allowed via CWD.
  stream = _secure_open(self._path, "rb")


46435 test instances
first 5 features: [{'words': 'SOCCER', 'pos': 'NN'}, {'words': '-', 'pos': ':'}, {'words': 'JAPAN', 'pos': 'NNP'}, {'words': 'GET', 'pos': 'VB'}, {'words': 'LUCKY', 'pos': 'NNP'}]
first 5 labels:   ['O', 'O', 'B-LOC', 'O', 'O']


**[2 points] b) provide descriptive statistics about the training and test data:**
* How many instances are in train and test?
* Provide a frequency distribution of the NERC labels, i.e., how many times does each NERC label occur?
* Discuss to what extent the training and test data is balanced (equal amount of instances for each NERC label) and to what extent the training and test data differ?

Tip: you can use the following `Counter` functionality to generate frequency list of a list:

In [3]:
from collections import Counter 

my_list=[1,2,1,3,2,5]
Counter(my_list)


Counter({1: 2, 2: 2, 3: 1, 5: 1})

In [4]:
print('Train:', len(training_gold_labels))
print('Test: ', len(test_gold_labels))

train_dist = Counter(training_gold_labels)
test_dist  = Counter(test_gold_labels)

print('\nTrain label distribution:')
for label, n in train_dist.most_common():
    print(f'  {label:8s} {n:>7d}  ({n/len(training_gold_labels):.2%})')

print('\nTest label distribution:')
for label, n in test_dist.most_common():
    print(f'  {label:8s} {n:>7d}  ({n/len(test_gold_labels):.2%})')


Train: 203621
Test:  46435

Train label distribution:
  O         169578  (83.28%)
  B-LOC       7140  (3.51%)
  B-PER       6600  (3.24%)
  B-ORG       6321  (3.10%)
  I-PER       4528  (2.22%)
  I-ORG       3704  (1.82%)
  B-MISC      3438  (1.69%)
  I-LOC       1157  (0.57%)
  I-MISC      1155  (0.57%)

Test label distribution:
  O          38323  (82.53%)
  B-LOC       1668  (3.59%)
  B-ORG       1661  (3.58%)
  B-PER       1617  (3.48%)
  I-PER       1156  (2.49%)
  I-ORG        835  (1.80%)
  B-MISC       702  (1.51%)
  I-LOC        257  (0.55%)
  I-MISC       216  (0.47%)


Data is highly imbalanced. O accounts for 83% of tokens and tests a similar amount. A classifier that alwyas predicts O would already score 83% accuracy making it a misleading metric for this task. Per class precision/recall matters. 

B-LOC, B-PER and B-ORG are the msot frequnet 3.5-3% each. While inside tags I-LOC and I-MISC are rare. THis skew means the classifier will see far fewer examples of inside-tags and struggle on them.

Finally, test and train are similar in shape. This is good it means the test set is a realistic and representative sample. Thus test scores should reasonaly reflect the models generalization.


**[2 points] c) Concatenate the train and test features (the list of dictionaries) into one list. Load it using the *DictVectorizer*. Afterwards, split it back to training and test.**

Tip: You’ve concatenated train and test into one list and then you’ve applied the DictVectorizer.
The order of the rows is maintained. You can hence use an index (number of training instances) to split the_array back into train and test. Do NOT use: `
from sklearn.model_selection import train_test_split` here.


In [5]:
from sklearn.feature_extraction import DictVectorizer

In [6]:
vec = DictVectorizer()
the_array = vec.fit_transform(training_features + test_features)

n_train = len(training_features)
X_train = the_array[:n_train]
X_test  = the_array[n_train:]

print('array:', the_array.shape)
print('X_train: ', X_train.shape)
print('X_test:  ', X_test.shape)

array: (250056, 27361)
X_train:  (203621, 27361)
X_test:   (46435, 27361)


**[4 points] d) Train the SVM using the train features and labels and evaluate on the test data. Provide a classification report (sklearn.metrics.classification_report).**
The train (*lin_clf.fit*) might take a while. On my computer, it took 1min 53s, which is acceptable. Training models normally takes much longer. If it takes more than 5 minutes, you can use a subset for training. Describe the results:
* Which NERC labels does the classifier perform well on? Why do you think this is the case?
* Which NERC labels does the classifier perform poorly on? Why do you think this is the case?

In [13]:
from sklearn import svm
from sklearn.metrics import classification_report


In [14]:
lin_clf = svm.LinearSVC()
lin_clf.fit(X_train, training_gold_labels)


c:\Users\nicol\anaconda3\envs\text_mining\lib\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


,penalty,'l2'
,loss,'squared_hinge'
,dual,'auto'
,tol,0.0001
,C,1.0
,multi_class,'ovr'
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,verbose,0
,random_state,None


In [15]:
preds = lin_clf.predict(X_test)
print(classification_report(test_gold_labels, preds, digits=3))

              precision    recall  f1-score   support

       B-LOC      0.809     0.773     0.791      1668
      B-MISC      0.776     0.658     0.712       702
       B-ORG      0.788     0.518     0.625      1661
       B-PER      0.865     0.436     0.580      1617
       I-LOC      0.618     0.529     0.570       257
      I-MISC      0.585     0.588     0.587       216
       I-ORG      0.660     0.477     0.554       835
       I-PER      0.330     0.870     0.479      1156
           O      0.986     0.984     0.985     38323

    accuracy                          0.919     46435
   macro avg      0.713     0.648     0.654     46435
weighted avg      0.939     0.919     0.922     46435



Overall accuracy is 92% which is misleading. Predicitng O always would already get 83%. The macro-F1 of 0.65 which is more informative as it shows performance varies across labels. 

Best:
O at F1 0.99. Expected as it maikes up 83% of tokens. Among entity tags B-LOC (0.79) and B-MISC (0.71) score highest. B-LOC benefits from being both frequent in training and lexically distinctive (capitalized proper nouns like Spain or Netherlands)

Worst:
B-PER has high precision 0.87 but low recall 0.44 F1 0.58. The model rearly guesses B-PER wrong but it misses many real persons. Likey because person names are unbounded vocabulary and many test names dont appear in train. I-PER shows the opposite low precision but very high recall. The classifier over-predicts I-PER expected given model can't see context features. I-LOC, I-MISC, I-ORG sit at 0.55-0.59 F1 hurt by their rarity and by also by lack of content. 

**[6 points] e) Train a model that uses the embeddings of these words as inputs. Test again on the same data as in 2d. Generate a classification report and compare the results with the classifier you built in 2d.**

In [16]:
# your code here
import gensim.downloader as api
import numpy as np

w2v = api.load('word2vec-google-news-300')


def to_vec(token):
    return w2v[token] if token in w2v else np.zeros(300)


X_train_emb = np.array([to_vec(f['words']) for f in training_features])
X_test_emb  = np.array([to_vec(f['words']) for f in test_features])


print('X_train_emb:', X_train_emb.shape)
print('X_test_emb: ', X_test_emb.shape)



lin_clf_emb = svm.LinearSVC(max_iter=2000)
lin_clf_emb.fit(X_train_emb, training_gold_labels)
preds_emb = lin_clf_emb.predict(X_test_emb)

print(classification_report(test_gold_labels, preds_emb, digits=3))



X_train_emb: (203621, 300)
X_test_emb:  (46435, 300)
              precision    recall  f1-score   support

       B-LOC      0.759     0.801     0.779      1668
      B-MISC      0.724     0.695     0.709       702
       B-ORG      0.690     0.638     0.663      1661
       B-PER      0.746     0.669     0.705      1617
       I-LOC      0.514     0.424     0.465       257
      I-MISC      0.604     0.537     0.569       216
       I-ORG      0.480     0.332     0.392       835
       I-PER      0.586     0.501     0.540      1156
           O      0.973     0.991     0.982     38323

    accuracy                          0.927     46435
   macro avg      0.675     0.621     0.645     46435
weighted avg      0.921     0.927     0.923     46435



Accuracy goes up slightly to 92.7% macro F1 drops a touch to 0.65. Pretty close to d overall but the per label is different.

Where embedings help:
B-PER jumps from 0.58 to 0.71 mainly from recall going from 0.44 to 0.67. Person names that didnt appear in train now sit close to known names in vector space so the model generalizes. B-ORG also improves for the same reason. I-PER becomes more balanced precision climbs from 0.33 to 0.59 while recall comes down to a sensible 0.50, the over-prediction problem from d is mostly fixed.

Where embeddings hurt:
Inside tags get worse. I-LOC drops from 0.57 to 0.47 and I-ORG from 0.55 to 0.39. These tags relied on POS info (Union inside European Union is NNP) and that signal is gone. O is basically the same at 0.98. B-MISC drops a bit because MISC was relying on surface patterns like nationality endings that embeddings smooth over.

The trade off is lexical sparsity for semantic generalization. Embeddings help open vocab entity types like persons and orgs but hurt inside tags and rare classes that needed POS. Macro F1 staying flat reflects that gains and losses cancel. A combined model with both one-hot POS and word embedding would probably beat both.

## [Points: 10] Exercise 2 (NERC): feature inspection using the [Annotated Corpus for Named Entity Recognition](https://www.kaggle.com/abhinavwalia95/entity-annotated-corpus)
**[6 points] a. Perform the same steps as in the previous exercise. Make sure you end up for both the training part (*df_train*) and the test part (*df_test*) with:**
* the features representation using **DictVectorizer**
* the NERC labels in a list

Please note that this is the same setup as in the previous exercise:
* load both train and test using:
    * list of dictionaries for features
    * list of NERC labels
* combine train and test features in a list and represent them using one hot encoding
* train using the training features and NERC labels

In [17]:
import pandas

In [21]:
##### Adapt the path to point to your local copy of NERC_datasets ok
path = r'kaggle/ner_v2.csv'
kaggle_dataset = pandas.read_csv(path, on_bad_lines='warn', encoding='latin1')


C:\Users\nicol\AppData\Local\Temp\ipykernel_42056\1741348650.py:3: ParserWarning: Skipping line 281837: expected 25 fields, saw 34

  kaggle_dataset = pandas.read_csv(path, on_bad_lines='warn', encoding='latin1')


In [22]:
len(kaggle_dataset)

1050795

In [23]:
df_train = kaggle_dataset[:100000]
df_test = kaggle_dataset[100000:120000]
print(len(df_train), len(df_test))

100000 20000


In [25]:
feature_cols = ['word', 'pos', 'lemma', 'prev-word', 'prev-pos',
                'next-word', 'next-pos', 'shape', 'prev-iob']

def to_features(df):
    feats = df[feature_cols].astype(str).to_dict(orient='records')
    labels = df['tag'].astype(str).tolist()
    return feats, labels

train_features, train_labels = to_features(df_train)
test_features,  test_labels  = to_features(df_test)

vec = DictVectorizer()

the_array = vec.fit_transform(train_features + test_features)
X_train = the_array[:len(train_features)]
X_test  = the_array[len(train_features):]

print('shapes:', X_train.shape, X_test.shape)



shapes: (100000, 43428) (20000, 43428)


**[4 points] b. Train and evaluate the model and provide the classification report:**
* use the SVM to predict NERC labels on the test data
* evaluate the performance of the SVM on the test data

Analyze the performance per NERC label.

In [26]:
clf = svm.LinearSVC(max_iter=2000)
clf.fit(X_train, train_labels)
preds = clf.predict(X_test)
print(classification_report(test_labels, preds, digits=3))



              precision    recall  f1-score   support

       B-art      0.333     0.250     0.286         4
       B-eve      0.000     0.000     0.000         0
       B-geo      0.854     0.870     0.862       741
       B-gpe      0.929     0.932     0.931       296
       B-nat      1.000     0.625     0.769         8
       B-org      0.761     0.683     0.720       397
       B-per      0.816     0.799     0.807       333
       B-tim      0.954     0.847     0.898       393
       I-geo      0.968     0.981     0.975       156
       I-gpe      1.000     1.000     1.000         2
       I-nat      1.000     1.000     1.000         4
       I-org      0.946     0.935     0.940       321
       I-per      0.951     0.981     0.966       319
       I-tim      0.960     0.898     0.928       108
           O      0.990     0.994     0.992     16918

    accuracy                          0.974     20000
   macro avg      0.831     0.786     0.805     20000
weighted avg      0.974   

c:\Users\nicol\anaconda3\envs\text_mining\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\nicol\anaconda3\envs\text_mining\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\nicol\anaconda3\envs\text_mining\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Way better than Ex 1. Accuracy 97.4% and macro F1 0.81 versus 0.65 in Ex 1d. The extra features (prev/next word,shape, prev-iob) do a lot of work here.

Best:
O at 0.99 same as before. The geographic and political tags are very strong. B-gpe 0.93, B-tim 0.90, B-geo 0.86. These are mostly closed-vocab proper nouns with consistent shape and POS so the model picks them up easily. Inside tags are all huge improvements over Ex 1 with I-org 0.94, I-per 0.97, I-tim 0.93. This is mostly the prev-iob feature doing the work, the model can see that the previous token was B-org so it knows the current one is likely I-org. Basically a cheating feature since at real inference time you wouldn't have gold previous tags but the rubric just says use the same pipeline.

Worst:
Rare tags. B-art has only 4 test examples and gets F1 0.29. B-nat has 8 examples and scores 0.77 with perfect precision but missing 3 of the 8. B-eve has 0 test support which is why sklearn throws the UndefinedMetricWarning, the metric just isnt meaningful with no samples. B-org at 0.72 is the weakest of the common entity tags, organization names are open-vocab and lexically diverse so the SVM struggles even with rich features.

Compared to Ex 1d the per-tag pattern is similar in shape (frequent and distinctive tags do well, rare and open-vocab ones struggle) but everything is shifted up. The prev-iob feature is doing most of the lifting. Without it the inside tags would be much weaker, closer to what we saw in Ex 1.

## End of this notebook